# LeetCode 3079 — Find the Sum of Encrypted Integers bằng Python

Link đề: https://leetcode.com/problems/find-the-sum-of-encrypted-integers/ · Độ khó: **Easy** · Chủ đề: Array, Math, String

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def sumOfEncryptedInt(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hàm `encrypt(x)` **thay mỗi chữ số của `x` bằng chữ số lớn nhất trong `x`**; cần tính **tổng `encrypt(x)` trên cả mảng**.
2. Nắm hai cách làm việc với **chữ số** của một số nguyên: **chuyển sang chuỗi** (`str(x)`) và **toán học** (`x % 10`, `x // 10`), kèm `max` trên chuỗi và phép nhân chuỗi.
3. Đi từ cách **làm từng bước theo chữ (list chữ số → thay → ghép → số)** → **dùng chuỗi gọn: `max(s) * len(s)`** → **toán học thuần, không chuỗi** → **bảng tra trước cho mọi giá trị `1..1000`**.
4. Hiểu mấu chốt: kết quả `encrypt(x)` chỉ phụ thuộc vào **hai con số: chữ số lớn nhất `d` và số chữ số `k`**; đó là số `d` lặp `k` lần (`ddd...d`).
5. Tránh các lỗi kinh điển: nhầm kiểu (`max(s)` là **ký tự**, không phải số), thay sai (chỉ chữ số đầu), mất `x` khi dùng `while x > 0`, dùng `/` thay `//`, nhầm với `1000`.

## 2. Đề bài & ví dụ

Ta định nghĩa hàm `encrypt` như sau: `encrypt(x)` thay **mỗi chữ số của `x`** bằng **chữ số lớn nhất trong `x`**.

Ví dụ: `encrypt(523) = 555` (chữ số lớn nhất của `523` là `5`), và `encrypt(213) = 333` (chữ số lớn nhất là `3`).

Cho mảng số nguyên `nums`. Hãy trả về **tổng của `encrypt(nums[i])`** với mọi phần tử.

**Ví dụ 1**

- Input: `nums = [1, 2, 3]`
- Output: `6`
- Giải thích: số một chữ số thì chữ số lớn nhất chính là nó: `encrypt(1) = 1`, `encrypt(2) = 2`, `encrypt(3) = 3`. Tổng `6`.

**Ví dụ 2**

- Input: `nums = [10, 21, 31]`
- Output: `66`
- Giải thích: `encrypt(10) = 11`, `encrypt(21) = 22`, `encrypt(31) = 33`. Tổng `11 + 22 + 33 = 66`.

**Ví dụ 3 (tự dựng)**

- Input: `nums = [523, 213]`
- Output: `888`
- Giải thích: `555 + 333 = 888`.

**Ràng buộc:** `1 <= nums.length <= 50`; `1 <= nums[i] <= 1000`.

In [1]:
from typing import List
from functools import lru_cache
import random

examples = [
    ([1, 2, 3], 6),
    ([10, 21, 31], 66),
    ([523, 213], 888),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 3] -> kỳ vọng 6
nums = [10, 21, 31] -> kỳ vọng 66
nums = [523, 213] -> kỳ vọng 888


## 3. Kiến thức nền

### 3.1. Số nguyên và chữ số: `str(x)`

`str(523)` cho chuỗi `"523"`. Chuỗi duyệt được từng ký tự (chữ số), và `len("523")` là **số chữ số**. Ngược lại `int("555")` đổi chuỗi về số. Lưu ý: ký tự `"5"` **không phải** số `5` (kiểu `str` khác kiểu `int`).

In [2]:
x = 523
s = str(x)
print(s, type(s))               # 523 <class 'str'>
print(len(s))                   # 3 : số chữ số
print(list(s))                  # ['5', '2', '3']
print(int("555") + 1)           # 556
print("5" == 5)                 # False : khác kiểu

523 <class 'str'>
3
['5', '2', '3']
556
False


### 3.2. `max` trên chuỗi và trên list

`max("523")` duyệt từng **ký tự** và so sánh theo thứ tự bảng mã. Với các ký tự chữ số `'0'..'9'`, thứ tự này **trùng với thứ tự số**, nên `max("523")` là `'5'` đúng như mong đợi (nhưng vẫn là **ký tự**). Muốn số thì dùng `int(max(s))` hoặc `max(int(c) for c in s)`.

In [3]:
print(max("523"))                       # '5' (ký tự)
print(int(max("523")))                  # 5   (số)
print(max(int(c) for c in "523"))       # 5
print(max("1000"), max("9"), max("102"))    # 1 9 2

5
5
5
1 9 2


### 3.3. Nhân chuỗi: `"5" * 3`

Trong Python, nhân một **chuỗi** với một **số nguyên** lặp chuỗi đó: `"5" * 3 == "555"`. Đây chính là cách tạo "số `5` lặp `3` lần" mà đề cần; sau đó `int("555")` cho số `555`.

In [4]:
print("5" * 3)                   # 555
print(int("5" * 3))              # 555 (số)
print("ab" * 2)                  # abab
print(int(max("213") * len("213")))      # 333

555
555
abab
333


### 3.4. Lấy chữ số bằng `% 10` và `// 10`

- `x % 10` là **chữ số cuối** của `x`.
- `x // 10` là `x` **bỏ chữ số cuối** (chia nguyên).

Lặp lại hai thao tác cho tới khi `x` thành `0` ta duyệt hết các chữ số (từ phải sang trái). Cẩn thận: `x / 10` (một dấu `/`) cho số thực `52.3`, **không** dùng được ở đây.

In [5]:
x = 523
print(x % 10, x // 10)         # 3 52
print(52 % 10, 52 // 10)       # 2 5
print(5 % 10, 5 // 10)         # 5 0
print(523 / 10)                # 52.3 (số thực! không dùng)

3 52
2 5
5 0
52.3


### 3.5. Dựng số lặp chữ số bằng toán: "repunit"

Số gồm `k` chữ số `1` (`1`, `11`, `111`, ...) gọi là **repunit**. Nhân nó với một chữ số `d` cho "`d` lặp `k` lần": `5 * 111 = 555`. Có thể dựng repunit bằng vòng lặp `rep = rep * 10 + 1`.

In [6]:
rep = 0
for _ in range(3):
    rep = rep * 10 + 1
    print(rep)                 # 1, 11, 111
print(5 * rep)                 # 555

1
11
111
555


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `x = 213`.

1. Các chữ số là `2, 1, 3`.
2. Chữ số lớn nhất là `3`.
3. Có `3` chữ số ⇒ thay cả ba chữ số bằng `3` ⇒ `333`.

Với `x = 1000`: các chữ số `1, 0, 0, 0`, lớn nhất là `1`, có `4` chữ số ⇒ `1111`.

Với `x = 7`: chỉ một chữ số, lớn nhất là `7` ⇒ `7`.

**Rút ra:** với mỗi `x` ta chỉ cần biết **chữ số lớn nhất `d`** và **số chữ số `k`**; kết quả là `d` viết `k` lần. Rồi cộng tất cả các kết quả lại.

In [7]:
for x in [213, 1000, 7, 523, 10]:
    digits = [int(c) for c in str(x)]
    d, k = max(digits), len(digits)
    print(f"x={x}: chữ số={digits}, lớn nhất d={d}, số chữ số k={k} -> {int(str(d) * k)}")

x=213: chữ số=[2, 1, 3], lớn nhất d=3, số chữ số k=3 -> 333
x=1000: chữ số=[1, 0, 0, 0], lớn nhất d=1, số chữ số k=4 -> 1111
x=7: chữ số=[7], lớn nhất d=7, số chữ số k=1 -> 7
x=523: chữ số=[5, 2, 3], lớn nhất d=5, số chữ số k=3 -> 555
x=10: chữ số=[1, 0], lớn nhất d=1, số chữ số k=2 -> 11


## 5. Cách 1 — Brute Force / baseline: làm từng bước theo đúng chữ của đề

**Ý tưởng.** Với mỗi số `x`:

1. Tách `x` thành **list chữ số** (kiểu số nguyên).
2. Tìm chữ số lớn nhất `d`.
3. Tạo list mới gồm `d` lặp lại đúng độ dài (thay **từng** chữ số bằng `d`, như đề nói).
4. Ghép các chữ số thành một chuỗi rồi `int(...)`.

Sau đó cộng kết quả cho mọi phần tử. Cách này dài nhưng **mỗi bước đều bám sát lời đề**.

In [8]:
def encrypt_literal(x: int) -> int:
    digits = [int(c) for c in str(x)]              # 1) tách chữ số
    biggest = max(digits)                          # 2) chữ số lớn nhất
    replaced = [biggest for _ in digits]           # 3) thay mỗi chữ số bằng biggest
    return int("".join(str(d) for d in replaced))  # 4) ghép lại thành số


def sum_encrypted_literal(nums: List[int]) -> int:
    total = 0
    for x in nums:
        total += encrypt_literal(x)
    return total


print(encrypt_literal(523), encrypt_literal(213), encrypt_literal(1000))     # 555 333 1111
for nums, expected in examples:
    print(sum_encrypted_literal(nums), "(kỳ vọng", expected, ")")

555 333 1111
6 (kỳ vọng 6 )
66 (kỳ vọng 66 )
888 (kỳ vọng 888 )


**Độ phức tạp (Cách 1).** Gọi `n` là số phần tử, `D` là số chữ số tối đa (ở đây `D <= 4`). Mỗi số xử lý `O(D)` ⇒ **Time `O(n · D)`**, **Space `O(D)`** (list chữ số tạm).

**Nút thắt:** bước 3 và 4 hơi thừa: ta biết kết quả là chữ số `d` lặp `k` lần, có thể tạo thẳng bằng nhân chuỗi.

## 6. Ý tưởng thuật toán / bước đột phá

`encrypt(x)` chỉ cần hai thông số:

- `d = max` chữ số của `x`;
- `k` = số chữ số của `x` (`len(str(x))`).

Kết quả là số gồm `k` chữ số `d`: `int(str(d) * k)`. Toán học thì là `d * (10^k - 1) / 9` (ví dụ `5 * 111 = 5 * (10^3 - 1) / 9 = 555`). Ta không cần dựng list thay thế.

Hơn nữa, vì `x <= 1000` nên chỉ có **1000 giá trị** có thể xuất hiện: có thể **tính trước** `encrypt` cho mọi `x` một lần (bảng tra).

In [9]:
# Công thức: encrypt(x) = d * (10^k - 1) // 9
for x in [523, 213, 1000, 7, 10]:
    s = str(x)
    d, k = int(max(s)), len(s)
    via_formula = d * (10 ** k - 1) // 9
    via_string = int(str(d) * k)
    print(x, via_formula, via_string)
    assert via_formula == via_string

523 555 555
213 333 333
1000 1111 1111
7 7 7
10 11 11


## 7. Các cách tiếp theo

### Cách 2 — Chuỗi gọn: `int(max(s) * len(s))`

**Giải thích từng đoạn.**

- `s = str(x)`: đổi sang chuỗi.
- `max(s)`: **ký tự** chữ số lớn nhất (đúng thứ tự vì là các ký tự `'0'..'9'`).
- `max(s) * len(s)`: lặp ký tự đó `len(s)` lần ⇒ chuỗi `"555"`.
- `int(...)`: đổi về số. Tổng bằng `sum(... for x in nums)`.

In [10]:
def encrypt_string(x: int) -> int:
    s = str(x)
    return int(max(s) * len(s))


def sum_encrypted_string(nums: List[int]) -> int:
    return sum(encrypt_string(x) for x in nums)


for nums, expected in examples:
    print(sum_encrypted_string(nums), "(kỳ vọng", expected, ")")

6 (kỳ vọng 6 )
66 (kỳ vọng 66 )
888 (kỳ vọng 888 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionString:
    def sumOfEncryptedInt(self, nums: List[int]) -> int:
        return sum(int(max(str(x)) * len(str(x))) for x in nums)


print(SolutionString().sumOfEncryptedInt([10, 21, 31]))      # 66

66


**Độ phức tạp Cách 2:** **Time `O(n · D)`**, **Space `O(D)`** (chuỗi tạm). Ngắn nhất, đọc là hiểu.

### Cách 3 — Toán học thuần (không chuỗi): `% 10`, `// 10`

**Giải thích từng đoạn.** Với mỗi `x`:

1. Khởi tạo `biggest = 0` (chữ số lớn nhất đã thấy), `repunit = 0` (số gồm các chữ số `1`).
2. `while x > 0`: lấy chữ số cuối `digit = x % 10`, cập nhật `biggest = max(biggest, digit)`; mỗi chữ số làm `repunit = repunit * 10 + 1` (thêm một chữ số `1`); rồi `x //= 10`.
3. Kết quả `biggest * repunit` (vd `5 * 111 = 555`).

Lưu ý: ta thay đổi biến `x` bên trong vòng lặp; vì thế dùng một bản sao (`y`) để không mất giá trị gốc nếu còn cần.

In [12]:
def encrypt_math(x: int) -> int:
    y = x
    biggest = 0
    repunit = 0
    while y > 0:
        digit = y % 10                         # chữ số cuối
        biggest = max(biggest, digit)          # cập nhật chữ số lớn nhất
        repunit = repunit * 10 + 1             # thêm một chữ số 1 (đếm số chữ số)
        y //= 10                               # bỏ chữ số cuối
    return biggest * repunit


def sum_encrypted_math(nums: List[int]) -> int:
    return sum(encrypt_math(x) for x in nums)


print(encrypt_math(523), encrypt_math(213), encrypt_math(1000))     # 555 333 1111
for nums, expected in examples:
    print(sum_encrypted_math(nums), "(kỳ vọng", expected, ")")

555 333 1111
6 (kỳ vọng 6 )
66 (kỳ vọng 66 )
888 (kỳ vọng 888 )


**Độ phức tạp Cách 3:** **Time `O(n · D)`**, **Space `O(1)`** (chỉ vài biến số). Không tạo chuỗi nào; là cách "chuẩn" trong các bài chữ số.

### Cách 4 — Bảng tra: tính trước `encrypt` cho `1..1000`

**Giải thích.** Mọi `nums[i] <= 1000` nên có đúng 1000 giá trị đầu vào có thể. Tính `table[x] = encrypt(x)` một lần, rồi mỗi phần tử chỉ tra bảng `O(1)`. Cách này đáng giá khi có **rất nhiều** truy vấn; ở đây `n <= 50` nên chỉ để luyện tư duy **tính trước (precompute)**. Dùng `lru_cache` là cách ghi nhớ tự động tương tự.

In [13]:
table = [0] * 1001
for v in range(1, 1001):
    table[v] = encrypt_math(v)


def sum_encrypted_table(nums: List[int]) -> int:
    return sum(table[x] for x in nums)


@lru_cache(maxsize=None)
def encrypt_cached(x: int) -> int:
    return int(max(str(x)) * len(str(x)))


def sum_encrypted_cached(nums: List[int]) -> int:
    return sum(encrypt_cached(x) for x in nums)


for nums, expected in examples:
    print(sum_encrypted_table(nums), sum_encrypted_cached(nums), "(kỳ vọng", expected, ")")

6 6 (kỳ vọng 6 )
66 66 (kỳ vọng 66 )
888 888 (kỳ vọng 888 )


**Độ phức tạp Cách 4:** tiền xử lý **`O(V · D)`** với `V = 1000`; mỗi truy vấn **`O(1)`** ⇒ tổng **`O(V · D + n)`**, **Space `O(V)`**. Với `n <= 50` thì không nhanh hơn, nhưng là mẫu "tính trước" hữu ích.

## 8. TRACE từng bước

Ta trace Cách 3 (toán học) trên `nums = [10, 21, 31]` để thấy từng chữ số bị "gọt" ra.

In [14]:
def trace_encrypt_math(x: int) -> int:
    y = x
    biggest = 0
    repunit = 0
    print(f"x = {x}")
    while y > 0:
        digit = y % 10
        biggest = max(biggest, digit)
        repunit = repunit * 10 + 1
        print(f"  y={y}: chữ số cuối={digit}, biggest={biggest}, repunit={repunit}")
        y //= 10
    result = biggest * repunit
    print(f"  -> encrypt({x}) = {biggest} * {repunit} = {result}")
    return result


total = 0
for x in [10, 21, 31]:
    total += trace_encrypt_math(x)
print("tổng =", total)
assert total == 66

x = 10
  y=10: chữ số cuối=0, biggest=0, repunit=1
  y=1: chữ số cuối=1, biggest=1, repunit=11
  -> encrypt(10) = 1 * 11 = 11
x = 21
  y=21: chữ số cuối=1, biggest=1, repunit=1
  y=2: chữ số cuối=2, biggest=2, repunit=11
  -> encrypt(21) = 2 * 11 = 22
x = 31
  y=31: chữ số cuối=1, biggest=1, repunit=1
  y=3: chữ số cuối=3, biggest=3, repunit=11
  -> encrypt(31) = 3 * 11 = 33
tổng = 66


In [15]:
def trace_string(nums: List[int]) -> int:
    total = 0
    for x in nums:
        s = str(x)
        d, k = max(s), len(s)
        enc = int(d * k)
        total += enc
        print(f"x={x}: s={s!r}, max={d!r}, len={k} -> {d * k!r} -> {enc}, tổng={total}")
    return total


assert trace_string([523, 213]) == 888

x=523: s='523', max='5', len=3 -> '555' -> 555, tổng=555
x=213: s='213', max='3', len=3 -> '333' -> 333, tổng=888


Bảng trace Cách 3 cho `x = 31`:

| bước | y | `y % 10` | biggest | repunit | y sau `//= 10` |
|---|---|---|---|---|---|
| 1 | 31 | 1 | 1 | 1 | 3 |
| 2 | 3 | 3 | 3 | 11 | 0 |

`y = 0` nên dừng. Kết quả `3 * 11 = 33`. Làm tương tự `10 -> 11`, `21 -> 22`; tổng `11 + 22 + 33 = 66`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (toán học, `O(1)` bộ nhớ phụ): không phụ thuộc chuỗi, ngắn gọn và an toàn.

In [16]:
from typing import List

class Solution:
    def sumOfEncryptedInt(self, nums: List[int]) -> int:
        total = 0
        for x in nums:
            biggest = 0                         # chữ số lớn nhất của x
            repunit = 0                         # 1, 11, 111, ... theo số chữ số của x
            while x > 0:
                biggest = max(biggest, x % 10)
                repunit = repunit * 10 + 1
                x //= 10
            total += biggest * repunit          # chữ số lớn nhất lặp lại đúng số chữ số
        return total


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.sumOfEncryptedInt(nums))

[1, 2, 3] -> 6
[10, 21, 31] -> 66
[523, 213] -> 888


## 10. Vì sao thuật toán đúng

- **Vòng `while` duyệt đủ chữ số:** mỗi lần `x //= 10` bỏ đi một chữ số, `x` giảm nghiêm ngặt về `0`, đúng `k` lần với `k` là số chữ số (vì `x >= 1`).
- **`biggest` đúng:** cập nhật bằng `max` với **mọi** chữ số cuối đã lấy, nên cuối cùng là chữ số lớn nhất.
- **`repunit` đúng:** bắt đầu `0`, mỗi vòng `repunit * 10 + 1` ⇒ sau `k` vòng là `11...1` (`k` chữ số `1`).
- **Kết quả đúng:** `biggest * repunit = d · 11...1 = dd...d` (vì `d` là chữ số `0..9` nên không có nhớ khi nhân). Đó chính là `encrypt(x)`.
- **Tổng:** cộng riêng từng phần tử đúng định nghĩa đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `D` = số chữ số tối đa (`<= 4` vì `nums[i] <= 1000`), `V = 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. List chữ số, thay, ghép | `O(n · D)` | `O(D)` |
| 2. `int(max(s) * len(s))` | `O(n · D)` | `O(D)` |
| 3. Toán học `% 10`, `// 10` | `O(n · D)` | `O(1)` |
| 4. Bảng tra / `lru_cache` | `O(V · D + n)` | `O(V)` |

Vì `D <= 4` là hằng số nhỏ nên thực tế mọi cách đều `O(n)`.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng. Ngoài ra kiểm tra **từng `encrypt(x)`** trên mọi `x` từ `1` tới `1000` bằng các cách khác nhau.

In [17]:
sol = Solution()
impls = {
    "literal": sum_encrypted_literal,
    "string": sum_encrypted_string,
    "math": sum_encrypted_math,
    "table": sum_encrypted_table,
    "cached": sum_encrypted_cached,
    "leetcode": sol.sumOfEncryptedInt,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        data = list(nums)
        actual = fn(data)
        assert actual == expected, (label, key, nums, actual, expected)
        assert data == nums, ("input bị sửa", key)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [18]:
check_all([1], 1, "một số một chữ số nhỏ nhất")
check_all([9], 9, "một số một chữ số lớn nhất")
check_all([10], 11, "10 -> 11 (chữ số 0 bị thay bằng 1)")
check_all([1000], 1111, "giá trị lớn nhất theo đề: 1000 -> 1111")
check_all([999], 999, "mọi chữ số đã là lớn nhất")
check_all([100], 111, "một chữ số 1 và hai chữ số 0")
check_all([90, 9, 99], 99 + 9 + 99, "90 -> 99; 9 -> 9; 99 -> 99")
check_all([123, 321, 213], 333 * 3, "cùng chữ số, thứ tự khác: đều 333")
check_all([1000] * 50, 1111 * 50, "n = 50, toàn giá trị lớn nhất")
check_all(list(range(1, 51)), sum(encrypt_string(v) for v in range(1, 51)), "1..50 theo cách chuỗi")

PASS một số một chữ số nhỏ nhất
PASS một số một chữ số lớn nhất
PASS 10 -> 11 (chữ số 0 bị thay bằng 1)
PASS giá trị lớn nhất theo đề: 1000 -> 1111
PASS mọi chữ số đã là lớn nhất
PASS một chữ số 1 và hai chữ số 0
PASS 90 -> 99; 9 -> 9; 99 -> 99
PASS cùng chữ số, thứ tự khác: đều 333
PASS n = 50, toàn giá trị lớn nhất
PASS 1..50 theo cách chuỗi


In [19]:
# encrypt(x) từng giá trị 1..1000: bốn cách tính phải khớp nhau, và khớp định nghĩa gốc (thay từng chữ số)
def encrypt_by_definition(x):
    digits = list(str(x))
    biggest = max(digits)
    return int("".join(biggest for _ in digits))


for x in range(1, 1001):
    ref = encrypt_by_definition(x)
    assert encrypt_literal(x) == encrypt_string(x) == encrypt_math(x) == table[x] == encrypt_cached(x) == ref, x
print("PASS encrypt(x) trên toàn bộ 1..1000")

PASS encrypt(x) trên toàn bộ 1..1000


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **chỉ thay chữ số đầu** (thay chữ số đầu bằng chữ số lớn nhất, giữ phần còn lại); (b) **cộng chuỗi thay vì số** (nối `"555" + "333"`).

In [20]:
def wrong_replace_first_only(nums):
    total = 0
    for x in nums:
        s = str(x)
        total += int(max(s) + s[1:])            # SAI: chỉ thay chữ số đầu
    return total


def wrong_concat(nums):
    total = ""
    for x in nums:
        s = str(x)
        total += max(s) * len(s)                # SAI: cộng chuỗi -> nối chuỗi
    return total


bad = [523, 213]
print(wrong_replace_first_only(bad), "!=", sum_encrypted_math(bad))
print(repr(wrong_concat(bad)), "!=", sum_encrypted_math(bad))
assert wrong_replace_first_only(bad) != sum_encrypted_math(bad)
assert wrong_concat(bad) != sum_encrypted_math(bad)

836 != 888
'555333' != 888


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `nums` đúng ràng buộc (`1 <= n <= 50`, `1 <= nums[i] <= 1000`), trộn số một/hai/ba/bốn chữ số. Mọi cách phải khớp, và khớp một tham chiếu tính bằng định nghĩa.

In [21]:
rng = random.Random(3079)
for trial in range(2000):
    n = rng.choice([1, 2, 5, 20, 50])
    mode = rng.choice(["small", "mid", "big", "mix"])
    hi = {"small": 9, "mid": 99, "big": 1000, "mix": 1000}[mode]
    nums = [rng.randint(1, hi) for _ in range(n)]
    assert 1 <= len(nums) <= 50 and all(1 <= v <= 1000 for v in nums)
    ref = sum(encrypt_by_definition(v) for v in nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 2000 input ngẫu nhiên đúng ràng buộc: mọi cách khớp định nghĩa")

PASS 2000 input ngẫu nhiên đúng ràng buộc: mọi cách khớp định nghĩa


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đổi kiểu: `max(s)` là ký tự, không phải số

In [22]:
def sai_quen_doi_kieu(x):
    s = str(x)
    d = max(s)                      # '5' (ký tự)
    return d * len(s)               # "555" (chuỗi, không phải số)


r = sai_quen_doi_kieu(523)
print(repr(r), type(r))             # '555' <class 'str'>
try:
    sum([sai_quen_doi_kieu(523), sai_quen_doi_kieu(213)])
except TypeError as e:
    print("TypeError:", e)
print("đúng:", Solution().sumOfEncryptedInt([523, 213]))

'555' <class 'str'>
TypeError: unsupported operand type(s) for +: 'int' and 'str'
đúng: 888


**Vì sao sai:** kết quả là **chuỗi** `"555"`; cộng các chuỗi hoặc cộng với số báo `TypeError` (hoặc nối chuỗi âm thầm cho kết quả sai). **Cách sửa:** bọc `int(...)`.

### Lỗi 2: Chỉ thay chữ số đầu (hoặc chỉ các chữ số khác chữ số lớn nhất)

In [23]:
def sai_thay_chu_so_dau(x):
    s = str(x)
    return int(max(s) + s[1:])


print("sai:", sai_thay_chu_so_dau(213), "| đúng:", Solution().sumOfEncryptedInt([213]))

sai: 313 | đúng: 333


**Vì sao sai:** đề thay **mọi** chữ số. Code trên cho `"3" + "13" = 313` thay vì `333`. **Cách sửa:** lặp chữ số lớn nhất `len(s)` lần.

### Lỗi 3: Dùng `x` rồi làm mất giá trị khi `while x > 0`

In [24]:
def sai_mat_x(x):
    biggest = 0
    while x > 0:
        biggest = max(biggest, x % 10)
        x //= 10                    # x bị gọt về 0
    # lúc này không còn biết x có mấy chữ số nữa!
    return biggest * (10 ** len(str(x)) - 1) // 9      # SAI: len(str(0)) == 1


print("sai:", sai_mat_x(523), "| đúng:", Solution().sumOfEncryptedInt([523]))

sai: 5 | đúng: 555


**Vì sao sai:** sau vòng lặp `x = 0` nên `len(str(x)) = 1`, không còn là số chữ số ban đầu (`3`). **Cách sửa:** đếm số chữ số **ngay trong vòng lặp** (`repunit = repunit * 10 + 1` hoặc `count += 1`), hoặc lưu bản sao `y = x`.

### Lỗi 4: Dùng `/` thay `//`

In [25]:
def sai_chia_thuc(x):
    count = 0
    while x >= 1:                   # cố bù cho việc chia thực
        x = x / 10                  # SAI: số thực, 523 / 10 = 52.3
        count += 1
    return count


print("sai (đếm số chữ số):", sai_chia_thuc(523), "| nhưng x % 10 trên số thực:", 52.3 % 10)
print("đúng:", Solution().sumOfEncryptedInt([523]))

sai (đếm số chữ số): 3 | nhưng x % 10 trên số thực: 2.299999999999997
đúng: 555


**Vì sao sai (rủi ro):** `/` cho số thực nên `x % 10` trả về phần thập phân (`2.3...`), không phải chữ số. Dù đếm được số chữ số trong ví dụ này, lấy **chữ số** thì sai. **Cách sửa:** luôn dùng `//` khi bỏ chữ số cuối.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "**thay mỗi chữ số bằng ...**" ⇒ cân nhắc xem kết quả có phụ thuộc vào ít thông số hơn không (ở đây: `d` và `k`).
- Làm việc với chữ số: **chuỗi** (`str`) cho gọn, **toán học** (`% 10`, `// 10`) cho `O(1)` bộ nhớ và khi đề cấm chuỗi.
- `d * (10**k - 1) // 9` là công thức "chữ số `d` lặp `k` lần"; `11…1` gọi là *repunit*.
- `max` trên chuỗi chữ số hoạt động đúng nhờ thứ tự bảng mã của `'0'..'9'`; nhưng nhớ **đổi kiểu** về số.
- Khi giá trị đầu vào nhỏ (`<= 1000`), có thể **tính trước** cho mọi giá trị.

## 15. Biến thể của bài

1. **Thay bằng chữ số nhỏ nhất** thay vì lớn nhất.
2. **Find the Sum of Encrypted Integers với số lớn** (`x` rất lớn): dùng chuỗi, không dùng toán (hoặc tính cả hai).
3. **Trả về mảng `encrypt` của từng phần tử** (không cộng).
4. **Đếm số `x` mà `encrypt(x) == x`** (các số có mọi chữ số bằng nhau).
5. **Add Digits (258)**, **Sum of Digits of String After Convert (1945)**: cùng chủ đề thao tác chữ số.

In [26]:
def encrypt_min(x):
    s = str(x)
    return int(min(s) * len(s))


def encrypt_all(nums):
    return [encrypt_string(x) for x in nums]


def count_fixed_points(nums):
    return sum(encrypt_string(x) == x for x in nums)


print(encrypt_min(523), encrypt_min(1000))               # 222 0
print(encrypt_all([10, 21, 31]))                         # [11, 22, 33]
print(count_fixed_points([1, 11, 12, 222, 100]))         # 1, 11, 222 -> 3
assert encrypt_min(523) == 222 and encrypt_min(1000) == 0
assert encrypt_all([10, 21, 31]) == [11, 22, 33]
assert count_fixed_points([1, 11, 12, 222, 100]) == 3

222 0
[11, 22, 33]
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. List chữ số, thay, ghép | đúng từng chữ của đề | `O(n · D)` | `O(D)` | học, kiểm chứng |
| 2. `int(max(s) * len(s))` | chuỗi và nhân chuỗi | `O(n · D)` | `O(D)` | **gọn nhất**, hợp để đọc |
| 3. Toán `% 10`, `// 10` | `biggest * repunit` | `O(n · D)` | `O(1)` | **nên nộp**, không dùng chuỗi |
| 4. Bảng tra / cache | tính trước `1..1000` | `O(V · D + n)` | `O(V)` | rất nhiều truy vấn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `max("4071")`, `"7" * 3`, `int("7" * 3)`, `Solution().sumOfEncryptedInt([20])`, `Solution().sumOfEncryptedInt([98, 12])`.

**Bài 2 (tính tay).** Tính `encrypt` của `406`, `1000`, `75`, `909` rồi cộng tổng.

**Bài 3 (điền TODO).** Hoàn thành `encrypt_todo(x)` theo cách toán học (Cách 3).

In [27]:
def encrypt_todo(x):
    biggest = 0
    repunit = 0
    # TODO 1: while x > 0
    #   TODO 2: lấy chữ số cuối bằng % 10, cập nhật biggest = max(...)
    #   TODO 3: repunit = repunit * 10 + 1
    #   TODO 4: bỏ chữ số cuối bằng // 10
    # TODO 5: return biggest * repunit
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `sum_encrypted_formula(nums)` dùng công thức `d * (10**k - 1) // 9` với `d = int(max(str(x)))`, `k = len(str(x))`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `sum_encrypted_min(nums)`: thay mỗi chữ số bằng chữ số **nhỏ nhất**, rồi cộng. (b) Viết `decrypt_candidates(y)` trả về **mọi `x` trong `1..1000`** mà `encrypt(x) == y` (ví dụ `y = 33` cho các số có chữ số lớn nhất `3` và hai chữ số).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'7'`, `"777"`, `777`, `22`, `99 + 22 = 121`. Bài 3: vòng `while`, bốn dòng bên trong.
- **Mức 2 (vừa):** Bài 2: `406 -> 666`, `1000 -> 1111`, `75 -> 77`, `909 -> 999`; tổng `666 + 1111 + 77 + 999 = 2853`. Bài 4: `(10**k - 1) // 9` là repunit `k` chữ số.
- **Mức 3 (sâu):** Bài 5(a): dùng `min(s)` thay `max(s)`; chú ý kết quả có thể là `0` (ví dụ `1000 -> 0000 -> 0`). 5(b): duyệt `x` từ `1` tới `1000` và lọc `encrypt_string(x) == y`; với `y = 33`, `x` có đúng hai chữ số, chữ số lớn nhất `3` ⇒ `30, 31, 32, 33, 13, 23`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert max("4071") == "7"
assert "7" * 3 == "777"
assert int("7" * 3) == 777
assert Solution().sumOfEncryptedInt([20]) == 22
assert Solution().sumOfEncryptedInt([98, 12]) == 99 + 22

# Bài 2
assert [encrypt_string(v) for v in (406, 1000, 75, 909)] == [666, 1111, 77, 999]
assert Solution().sumOfEncryptedInt([406, 1000, 75, 909]) == 2853


# Bài 3
def encrypt_todo_ans(x):
    biggest = 0
    repunit = 0
    while x > 0:
        biggest = max(biggest, x % 10)
        repunit = repunit * 10 + 1
        x //= 10
    return biggest * repunit


for v in range(1, 1001):
    assert encrypt_todo_ans(v) == encrypt_string(v)


# Bài 4
def sum_encrypted_formula(nums):
    total = 0
    for x in nums:
        s = str(x)
        d, k = int(max(s)), len(s)
        total += d * (10 ** k - 1) // 9
    return total


for nums, expected in examples:
    assert sum_encrypted_formula(nums) == expected


# Bài 5(a)
def sum_encrypted_min(nums):
    return sum(encrypt_min(x) for x in nums)


assert sum_encrypted_min([523, 1000, 71]) == 222 + 0 + 11


# Bài 5(b)
def decrypt_candidates(y):
    return [x for x in range(1, 1001) if encrypt_string(x) == y]


assert decrypt_candidates(33) == [13, 23, 30, 31, 32, 33]
assert decrypt_candidates(7) == [7]
assert decrypt_candidates(1111) == [1000]
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thay mỗi chữ số bằng chữ số lớn nhất rồi cộng)

```python
from typing import List

class Solution:
    def sumOfEncryptedInt(self, nums: List[int]) -> int:
        total = 0
        for x in nums:
            biggest = repunit = 0
            while x > 0:
                biggest = max(biggest, x % 10)
                repunit = repunit * 10 + 1
                x //= 10
            total += biggest * repunit
        return total
```

**Checklist**

- [ ] Mỗi chữ số đều bị thay (kết quả là `d` lặp `k` lần), không chỉ chữ số đầu?
- [ ] Nếu dùng chuỗi: đã `int(...)` kết quả `max(s) * len(s)` trước khi cộng?
- [ ] Nếu dùng toán: dùng `//` (không phải `/`) và đếm số chữ số **trong** vòng lặp?
- [ ] Đã thử `10 -> 11`, `1000 -> 1111`, số một chữ số?
- [ ] Không sửa `nums` gốc (biến vòng lặp `x` chỉ là bản sao giá trị)?
- [ ] Skeleton đúng `class Solution` / `def sumOfEncryptedInt(self, nums: List[int]) -> int`?